# Conjoint analysis for choice modeling
### Jimmy Azar

## Introduction

We'll perform choice-based conjoint analysis in order to study customer preferences for attributes of a given type of product. The product in this case is a tablet. A survey was conducted in which customers were asked to select one from among three alternatives presented to them. The attributes are of course varied across the alternatives. Every customer was shown 15 such choice sets (each with 3 alternatives), and therefore 15 selections were recorded.

The attributes of a tablet are: brand, screen size (in inches), HD storage, RAM capacity, battery life, and price (in dollars). We would like to understand the importance of these attributes for customers. 

## Data exploration

We load and inspect the data. The first 3 rows show that the customer with id = 1 selected the first alternative when presented with the first choice set or question. This is indicated by a "1" in the variable "choice". Therefore, this customer prefers an iPad tablet with a 7-inch screen, 32GB storage, 4GB ram, 7-hour battery life, and a price of \$499 over a Surface tablet or a Kindle tablet.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# data exploration
path_to_file = '../data/tablets.txt'
data = pd.read_csv(path_to_file)
data.head(6)

# data.shape: 137*15*3 x 10

,consumer_id,choiceset_id,alternative_id,choice,brand,size,storage,ram,battery,price
0,1,1,1,1,iPad,sz7inch,st32gb,r4gb,b7h,499
1,1,1,2,0,Surface,sz10inch,st64gb,r2gb,b9h,399
2,1,1,3,0,Kindle,sz9inch,st16gb,r2gb,b8h,499
3,1,2,1,1,iPad,sz8inch,st32gb,r1gb,b8h,399
4,1,2,2,0,Surface,sz10inch,st128gb,r4gb,b7h,299
5,1,2,3,0,Nexus,sz7inch,st64gb,r1gb,b9h,199


## Multinomial logit model

We'll build a conjoint multinomial model and train it over the data.

In [2]:
from statsmodels.formula.api import mnlogit

model = mnlogit('choice ~ 0 + brand + size + storage + ram + battery + price', data).fit()
model.summary()

Optimization terminated successfully.
         Current function value: 0.579977
         Iterations 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                 choice   No. Observations:                 6165
Model:                        MNLogit   Df Residuals:                     6149
Method:                           MLE   Df Model:                           15
Date:                Fri, 10 May 2024   Pseudo R-squ.:                 0.08882
Time:                        09:26:18   Log-Likelihood:                -3575.6
converged:                       True   LL-Null:                       -3924.1
Covariance Type:            nonrobust   LLR p-value:                7.721e-139
=====================================================================================
         choice=1       coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
brand[Galaxy]         0.5084      0.132      3.864      0.000       0.251       0.766
brand[Kindle]         0.5392      0.136      3.952      0.000       0.272       0.807
brand[Nexus]          0.1019      0.138      0.738      0.461      -0.169       0.373
brand[Surface]        0.3338      0.134      2.491      0.013       0.071       0.596
brand[iPad]           1.1690      0.142      8.228      0.000       0.891       1.447
size[T.sz7inch]      -0.2172      0.084     -2.578      0.010      -0.382      -0.052
size[T.sz8inch]      -0.1337      0.087     -1.529      0.126      -0.305       0.038
size[T.sz9inch]       0.1223      0.085      1.434      0.152      -0.045       0.290
storage[T.st16gb]    -0.6659      0.089     -7.455      0.000      -0.841      -0.491
storage[T.st32gb]    -0.3786      0.082     -4.591      0.000      -0.540      -0.217
storage[T.st64gb]    -0.0679      0.082     -0.824      0.410      -0.229       0.094
ram[T.r2gb]           0.3073      0.074      4.156      0.000       0.162       0.452
ram[T.r4gb]           0.8265      0.072     11.428      0.000       0.685       0.968
battery[T.b8h]        0.2859      0.074      3.845      0.000       0.140       0.432
battery[T.b9h]        0.2370      0.074      3.219      0.001       0.093       0.381
price                -0.0051      0.000    -19.521      0.000      -0.006      -0.005
=====================================================================================
"""

## Parameter interpretation 

The previous summary reveals the coefficients associated with each attribute level. These coefficients can be compared (in sign and magnitude). For example, price has a negative coefficient as expected. Note that the attribute levels not appearing in the summary are reference levels and their coefficients are zero. 

## Prediction

Next, we predict the probability of selecting each alternative for the given choice sets. This way, we can make customer choice predictions for new alternatives (and their competition) before we release these into the market. However, we would also like to estimate the market share based on these choice predictions. For that purpose, we may use the softmax function to squash the outputs into the range $[0,1]$ such that the sum equals 1.

If $U_j$ represents the (soft) label returned by the model for selecting some alternative $j$, then with $n$ alternatives being compared, the probability of selecting alternative $j$ is $p_j = \dfrac{exp(U_j)}{exp(U_1)+exp(U_2)+...+exp(U_n)}$, $j \in \{1,2,...,n\}$ (softmax)

The probabilities sum up to 1: $(p_1+p_2+...+p_n=1)$

We want to test out the market shares for 4 alternatives which we select from the original data. Finally, we change the amount of RAM for the iPad to 4GB instead of 2GB and compute the predicted market shares for the four competing alternatives in the new dataset. 

In [3]:
data_market = data.loc[[21,22,45,85],['brand','size','storage','ram','battery','price']].copy() 
data_market

,brand,size,storage,ram,battery,price
21,Surface,sz8inch,st128gb,r4gb,b8h,499
22,Kindle,sz8inch,st32gb,r1gb,b7h,199
45,Galaxy,sz8inch,st64gb,r4gb,b7h,499
85,iPad,sz7inch,st64gb,r2gb,b8h,499


In [4]:
# prediction
def predict_share(model, data):
    df = data.copy()
    df['alternative'] = model.predict(df)[1]
    df['alternative'] = np.exp(df['alternative'])
    df['alternative'] = df['alternative']/df['alternative'].sum() 
    return df['alternative']
   
data_market = data.loc[[21,22,45,85],['brand','size','storage','ram','battery','price']].copy() 
data_market['predicted_share'] = predict_share(model, data_market)
data_market

data_new = data_market.copy()
data_new.loc[85,'ram'] = 'r4gb'
data_new['predicted_share'] = predict_share(model, data_new)
print(data_new)

      brand     size  storage   ram battery  price  predicted_share
21  Surface  sz8inch  st128gb  r4gb     b8h    499         0.239956
22   Kindle  sz8inch   st32gb  r1gb     b7h    199         0.250858
45   Galaxy  sz8inch   st64gb  r4gb     b7h    499         0.232869
85     iPad  sz7inch   st64gb  r4gb     b8h    499         0.276316


## Willingness-to-pay

From the parameter (coefficient) estimates of the model, we can compute how much a consumer would be willing to pay for a specific attribute level, such as 9-hour battery life. We need to calculate the ratio of the coefficient for the specific attribute level to the coefficient of price (in absolute value). 

We compute the average consumer's willingness to pay (in dollars) to change a tablet brand from Galaxy to iPad (all else fixed). We also compute how much a consumer is willing to pay for an upgrade from 1GB to 4GB RAM, and an upgrade from a 7-inch screen to a 9-inch screen.

In [5]:
# willingness-to-pay
model.params

galaxy_to_ipad = (model.params.loc['brand[Galaxy]'] - model.params.loc['brand[iPad]'])/model.params.loc['price']
print(galaxy_to_ipad[0])

r1gb_to_r4gb = (0 - model.params.loc['ram[T.r4gb]'])/model.params.loc['price']
print(r1gb_to_r4gb[0])

sz7inch_to_sz9inch = (model.params.loc["size[T.sz7inch]"] - model.params.loc["size[T.sz9inch]"])/model.params.loc["price"]
print(sz7inch_to_sz9inch[0])

130.1177222425269
162.79756551253504
66.8848813260555
